# ClauseXplain: LegalBERT classification baseline (CUAD folds 0-2 + Indian set)
Trains `nlpaueb/legal-bert-base-uncased` as a classifier only (`cls_only`) on the SAME CUAD folds as the Longformer runs
(the saved `splits.json` is reused), with the same epochs / learning rates / seed, then scores the Indian main set
(334 clauses) and the judgment slice. No span head: BERT only takes 512 tokens.

Settings: attach `clausexplain-indian-eval` and `scan-vs-span`; **GPU T4**; **Internet ON** (downloads LegalBERT from
Hugging Face). Run all, then download `legalbert_results.zip` from the Output tab.
Roughly 15-30 min per fold. Set FOLDS = [0] for a quick first check.

In [1]:
import os, shutil, base64, io, zipfile, subprocess, sys, json
BASE = "/kaggle/input/datasets/riyasuryawanshi"
if not os.path.exists(BASE): BASE = "/kaggle/input"   # older Kaggle mount layout
EVAL_SRC, MODEL_SRC = f"{BASE}/clausexplain-indian-eval", f"{BASE}/scan-vs-span"
WORK, RUN = "/kaggle/working/clausexplain", "/kaggle/working/legalbert"
ENCODER = "nlpaueb/legal-bert-base-uncased"
FOLDS = [0, 1, 2]
for d in (WORK, RUN):
    if os.path.exists(d): shutil.rmtree(d)
shutil.copytree(EVAL_SRC, WORK)
zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))).extractall(WORK)   # current code + Indian set
os.makedirs(f"{RUN}/partial")
shutil.copy(f"{WORK}/partial/splits.json", f"{RUN}/partial/splits.json")   # identical CUAD folds
LF = json.load(open(f"{WORK}/partial/config.json"))
print("Longformer run settings reused:", {k: LF.get(k) for k in ["epochs", "lr", "head_lr", "seed", "test_frac", "val_frac",
                                                                 "bs_clause", "grad_accum", "class_weights", "max_len_clause"]})
assert os.path.exists(f"{MODEL_SRC}/CUAD_v1.json"), "attach the scan-vs-span dataset"
from transformers import AutoTokenizer
AutoTokenizer.from_pretrained(ENCODER)   # fails here if Internet is off
print("LegalBERT reachable")

Longformer run settings reused: {'epochs': 2, 'lr': 2e-05, 'head_lr': 0.0001, 'seed': 13, 'test_frac': 0.2, 'val_frac': 0.1, 'bs_clause': 16, 'grad_accum': 1, 'class_weights': 'none', 'max_len_clause': 512}


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

LegalBERT reachable


In [2]:
os.chdir(WORK)
opt = lambda k, flag=None: [f"--{flag or k}", str(LF[k])] if LF.get(k) is not None else []
cmd = [sys.executable, "train_phase3.py", "--cuad_json", f"{MODEL_SRC}/CUAD_v1.json",
       "--mapped_jsonl", f"{MODEL_SRC}/clauses_mapped.jsonl", "--out_dir", RUN, "--encoder", ENCODER,
       "--experiments", "cls_only", "--cv_folds", "5", "--folds", *map(str, FOLDS),
       "--max_len_span", "512", "--pad_multiple", "64", "--resume", "--save_ckpt", "--skip_baselines"]
for k in ["epochs", "lr", "head_lr", "seed", "test_frac", "val_frac", "bs_clause", "grad_accum", "class_weights", "max_len_clause"]:
    cmd += opt(k)
print(" ".join(cmd))
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "it/s" not in line and "s/it" not in line: print(line, end="")
assert p.wait() == 0, "training failed (see log above)"

/usr/bin/python3 train_phase3.py --cuad_json /kaggle/input/datasets/riyasuryawanshi/scan-vs-span/CUAD_v1.json --mapped_jsonl /kaggle/input/datasets/riyasuryawanshi/scan-vs-span/clauses_mapped.jsonl --out_dir /kaggle/working/legalbert --encoder nlpaueb/legal-bert-base-uncased --experiments cls_only --cv_folds 5 --folds 0 1 2 --max_len_span 512 --pad_multiple 64 --resume --save_ckpt --skip_baselines --epochs 2 --lr 2e-05 --head_lr 0.0001 --seed 13 --test_frac 0.2 --val_frac 0.1 --bs_clause 16 --grad_accum 1 --class_weights none --max_len_clause 512
device=cuda amp=torch.float16
contract text attached + offsets verified: {'verified': 13823}
9786 examples from 481 contracts; [('Intellectual Property', 1955), ('Termination', 1495), ('Liability', 1399), ('Commercial Terms', 1078), ('Other', 1075), ('Assignment', 907), ('Non-Compete', 794), ('Governing Law / Jurisdiction', 464), ('Warranty', 176), ('Dispute Resolution', 173), ('Non-Solicitation', 149), ('Penalties', 121)]
splits loaded from p

In [3]:
# CUAD test numbers per fold (compare: Longformer cls_only 0.839 macro-F1, TF-IDF 0.802, 3-fold means)
import statistics as st
rows = [json.load(open(f"{RUN}/partial/cls_only_fold{k}.json"))["cls"] for k in FOLDS]
for k, c in zip(FOLDS, rows):
    print(f"fold {k}: macro-F1 {c['macro_f1']:.3f}  micro-F1 {c['micro_f1']:.3f}  n {c['n']}")
print(f"mean: macro-F1 {st.mean(c['macro_f1'] for c in rows):.3f}  micro-F1 {st.mean(c['micro_f1'] for c in rows):.3f}")

fold 0: macro-F1 0.861  micro-F1 0.879  n 1983
fold 1: macro-F1 0.880  micro-F1 0.903  n 2043
fold 2: macro-F1 0.870  micro-F1 0.890  n 2070
mean: macro-F1 0.870  micro-F1 0.891


In [4]:
SETS = {"main": "built_v3", "judgments": "built_judgments"}
for k in FOLDS:
    for tag, src in SETS.items():
        out = f"{WORK}/{tag}_fold{k}"
        shutil.copytree(f"{WORK}/{src}", out, dirs_exist_ok=True)
        cmd = [sys.executable, "eval_indian.py", "--run_dir", RUN, "--fold", str(k), "--cls_only",
               "--ckpt", f"{RUN}/ckpt_cls_only_fold{k}.pt", "--indian_dir", out,
               "--mapped_jsonl", f"{MODEL_SRC}/clauses_mapped.jsonl", "--cuad_json", f"{MODEL_SRC}/CUAD_v1.json",
               "--clause_batch", "16"]
        print("=== fold", k, tag, "==="); r = subprocess.run(cmd, capture_output=True, text=True)
        print(r.stdout[-200:], r.stderr[-1500:] if r.returncode else "")

=== fold 0 main ===
x6, Intellectual Property x1 |

## Reference: CUAD cross-validated test numbers (mean over folds)

| Model | macro-F1 | micro-F1 | span token-F1 |
|---|---|---|---|
| cls_only | 0.870 | 0.891 | n/a |
 
=== fold 0 judgments ===
636 | 0.354 to 0.848 | 0.310 |

## Reference: CUAD cross-validated test numbers (mean over folds)

| Model | macro-F1 | micro-F1 | span token-F1 |
|---|---|---|---|
| cls_only | 0.870 | 0.891 | n/a |
 
=== fold 1 main ===
tion x1, Commercial Terms x1 |

## Reference: CUAD cross-validated test numbers (mean over folds)

| Model | macro-F1 | micro-F1 | span token-F1 |
|---|---|---|---|
| cls_only | 0.870 | 0.891 | n/a |
 
=== fold 1 judgments ===
636 | 0.354 to 0.848 | 0.454 |

## Reference: CUAD cross-validated test numbers (mean over folds)

| Model | macro-F1 | micro-F1 | span token-F1 |
|---|---|---|---|
| cls_only | 0.870 | 0.891 | n/a |
 
=== fold 2 main ===
 Termination x2, Warranty x1 |

## Reference: CUAD cross-validated test numbers (m

In [5]:
# LegalBERT accuracy on seen categories by source (Longformer v3 means: all 0.344, original 0.377, batch 2 0.275, judgments 0.606)
import csv, math
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),) * 2
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d; h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return max(0, c-h), min(1, c+h)
BATCH2 = ("indian_cpwd_gcc_2023", "indian_sbm_waste_to_compost_mca")
print("| fold | slice | n | accuracy | 95% interval | macro-F1 (whole set) |\n|---|---|---|---|---|---|")
for k in FOLDS:
    for tag in ["main", "judgments"]:
        rows = [r for r in csv.DictReader(open(f"{WORK}/{tag}_fold{k}/indian_predictions.csv", encoding="utf-8")) if r["seen_category"] == "1"]
        mf1 = json.load(open(f"{WORK}/{tag}_fold{k}/indian_eval.json"))["cls"]["macro_f1"]
        sl = {"all 334 (main)": rows, "original 228": [r for r in rows if r["document"] not in BATCH2],
              "batch 2 only": [r for r in rows if r["document"] in BATCH2]} if tag == "main" else {"judgments": rows}
        for name, rs in sl.items():
            n = len(rs); c = sum(int(r["correct"]) for r in rs); lo, hi = wilson(c, n)
            extra = f"{mf1:.3f}" if name in ("all 334 (main)", "judgments") else ""
            print(f"| {k} | {name} | {n} | " + (f"{c/n:.3f} | {lo:.3f} to {hi:.3f} | {extra} |" if n else "n/a | n/a | |"))

| fold | slice | n | accuracy | 95% interval | macro-F1 (whole set) |
|---|---|---|---|---|---|
| 0 | all 334 (main) | 192 | 0.375 | 0.310 to 0.445 | 0.388 |
| 0 | original 228 | 129 | 0.419 | 0.337 to 0.505 |  |
| 0 | batch 2 only | 63 | 0.286 | 0.189 to 0.407 |  |
| 0 | judgments | 11 | 0.545 | 0.280 to 0.787 | 0.229 |
| 1 | all 334 (main) | 192 | 0.323 | 0.261 to 0.392 | 0.301 |
| 1 | original 228 | 129 | 0.341 | 0.265 to 0.426 |  |
| 1 | batch 2 only | 63 | 0.286 | 0.189 to 0.407 |  |
| 1 | judgments | 11 | 0.545 | 0.280 to 0.787 | 0.229 |
| 2 | all 334 (main) | 192 | 0.349 | 0.285 to 0.419 | 0.335 |
| 2 | original 228 | 129 | 0.364 | 0.286 to 0.450 |  |
| 2 | batch 2 only | 63 | 0.317 | 0.216 to 0.440 |  |
| 2 | judgments | 11 | 0.545 | 0.280 to 0.787 | 0.229 |


In [6]:
with zipfile.ZipFile("/kaggle/working/legalbert_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["results.json", "results.md", "partial/config.json"] + [f"partial/cls_only_fold{k}.json" for k in FOLDS]:
        if os.path.exists(f"{RUN}/{f}"): z.write(f"{RUN}/{f}", f"cuad/{f}")
    for k in FOLDS:
        for tag in ["main", "judgments"]:
            for f in ["indian_eval.json", "indian_eval.md", "indian_predictions.csv"]:
                p = f"{WORK}/{tag}_fold{k}/{f}"
                if os.path.exists(p): z.write(p, f"{tag}/fold{k}/{f}")
print("download /kaggle/working/legalbert_results.zip from the Output tab (checkpoints are not included)")

download /kaggle/working/legalbert_results.zip from the Output tab (checkpoints are not included)
